# Evaluator ablation: count and position judged by Qwen instead of OWLv2

The main pipeline labels count (N) and position (P) with OWLv2 detections and all other constraints with 8-bit Qwen2.5-VL-7B. Qwen was also asked the N and P questions for every image, and its parsed answers are stored in `labels.csv` (`qwen_N`, `qwen_P`). This notebook replaces the OWLv2 labels for N and P with the Qwen labels and recomputes the metrics for SD 1.5 and SDXL, to check whether the conclusions depend on the evaluator choice.

- No model is run; CPU runtime is enough.
- Same rules as the pipeline: if the main object is judged absent, every constraint of the image fails; images with an unknown object label are excluded.
- Interference: same-seed minimal pairs where the object is present in both images; exact McNemar test with Holm correction over the 20 (A, B) pairs; 95% bootstrap interval over objects (2,000 resamples).
- Agreement between OWLv2 and Qwen labels is reported with Cohen's κ.

Inputs: `labels.csv`, `prompts.csv` and `pairs.csv` of `main_sd15` and of the complete SDXL run (`main_sdxl 2`). If a folder is not found, the notebook asks to upload these three files.

## 1. Inputs

In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display
TOOLS = Path("/content/ablation_tools")
TOOLS.mkdir(exist_ok=True)
(TOOLS / "ablation.py").write_text("import csv, json, math, random\nfrom pathlib import Path\n\nFACTORS = ['C', 'N', 'P', 'M', 'O2']\n\n\ndef b(v):\n    return {'True': True, 'False': False}.get(v)\n\n\ndef load(root):\n    root = Path(root)\n    rd = lambda n: list(csv.DictReader(open(root / n, newline='', encoding='utf-8')))\n    prompts = {p['prompt_id']: p for p in rd('prompts.csv')}\n    for p in prompts.values():\n        p['active'] = p['active_factors'].split('|') if p['active_factors'] else []\n    labels = rd('labels.csv')\n    for r in labels:\n        r['seed'] = int(r['seed'])\n    return prompts, labels, rd('pairs.csv')\n\n\ndef variant(prompts, labels, evaluator):\n    out = {}\n    for r in labels:\n        p = prompts[r['prompt_id']]\n        obj = b(r['object'])\n        row = {'object': obj, 'object_name': p['object'], 'complexity': int(p['complexity']), 'prompt_id': r['prompt_id'], 'seed': r['seed']}\n        for f in FACTORS:\n            if f not in p['active']:\n                continue\n            src = f'qwen_{f}' if evaluator == 'qwen' and f in {'N', 'P'} else f\n            v = b(r[src])\n            if obj is False:\n                v = False\n            elif obj is None:\n                v = None\n            row[f] = v\n        out[(r['prompt_id'], r['seed'])] = row\n    return out\n\n\ndef mcnemar(b_, c_):\n    n = b_ + c_\n    if n == 0:\n        return 1.0\n    k = min(b_, c_)\n    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)\n\n\ndef holm(ps):\n    order = sorted(range(len(ps)), key=lambda i: ps[i])\n    adj, running = [None] * len(ps), 0\n    for rank, i in enumerate(order):\n        running = max(running, min(1.0, (len(ps) - rank) * ps[i]))\n        adj[i] = running\n    return adj\n\n\ndef boot_ci(items, stat, draws=2000, seed=0):\n    groups = {}\n    for obj, x in items:\n        groups.setdefault(obj, []).append(x)\n    keys = sorted(groups)\n    if not keys:\n        return None, None\n    rng = random.Random(seed)\n    vals = []\n    for _ in range(draws):\n        sample = [x for k in (rng.choice(keys) for _ in keys) for x in groups[k]]\n        v = stat(sample)\n        if v is not None:\n            vals.append(v)\n    vals.sort()\n    if not vals:\n        return None, None\n    return vals[int(0.025 * len(vals))], vals[min(len(vals) - 1, int(0.975 * len(vals)))]\n\n\ndef acc(vals):\n    known = [v for v in vals if v is not None]\n    return sum(known) / len(known) if known else None\n\n\ndef attributes(lab):\n    rows = []\n    for f in ['object'] + FACTORS:\n        vals = [r.get(f) for r in lab.values() if f in r]\n        single = [r.get(f) for r in lab.values() if f in r and r['complexity'] == 1]\n        rows.append({'factor': f, 'n': sum(v is not None for v in vals), 'accuracy': acc(vals), 'singleton_accuracy': acc(single)})\n    return rows\n\n\ndef strict(lab, prompts):\n    out = {}\n    for (pid, s), r in lab.items():\n        vals = [r.get(f) for f in ['object'] + prompts[pid]['active']]\n        out[(pid, s)] = all(vals) if all(v is not None for v in vals) else None\n    return out\n\n\ndef by_complexity(lab, prompts):\n    st = strict(lab, prompts)\n    rows = []\n    for c in range(6):\n        vals = [v for k, v in st.items() if lab[k]['complexity'] == c]\n        rows.append({'complexity': c, 'n': sum(v is not None for v in vals), 'strict_accuracy': acc(vals)})\n    rows.append({'complexity': 'all', 'n': sum(v is not None for v in st.values()), 'strict_accuracy': acc(list(st.values()))})\n    return rows\n\n\ndef interference(lab, prompts, pairs):\n    rows = []\n    for a in FACTORS:\n        for bf in FACTORS:\n            if a == bf:\n                continue\n            comps = []\n            for pr in pairs:\n                if pr['added_factor'] != bf or a not in prompts[pr['base_id']]['active']:\n                    continue\n                for s in range(4):\n                    x, y = lab.get((pr['base_id'], s)), lab.get((pr['added_id'], s))\n                    if x and y and x['object'] is True and y['object'] is True and x.get(a) is not None and y.get(a) is not None:\n                        comps.append((x['object_name'], (x[a], y[a])))\n            n = len(comps)\n            lost = sum(x and not y for _, (x, y) in comps)\n            gained = sum(y and not x for _, (x, y) in comps)\n            delta = (gained - lost) / n if n else None\n            lo, hi = boot_ci(comps, lambda s: sum(int(y) - int(x) for x, y in s) / len(s) if s else None)\n            rows.append({'checked': a, 'added': bf, 'pairs': n, 'delta': delta, 'ci_low': lo, 'ci_high': hi, 'p': mcnemar(lost, gained)})\n    for r, adj in zip(rows, holm([r['p'] for r in rows])):\n        r['p_holm'] = adj\n    return rows\n\n\ndef agreement(lab_owl, lab_qwen, factor):\n    a = [(lab_owl[k][factor], lab_qwen[k][factor]) for k in lab_owl\n         if factor in lab_owl[k] and lab_owl[k]['object'] is True and lab_owl[k][factor] is not None and lab_qwen[k].get(factor) is not None]\n    n = len(a)\n    if not n:\n        return {'factor': factor, 'n': 0}\n    po = sum(x == y for x, y in a) / n\n    p1, p2 = sum(x for x, _ in a) / n, sum(y for _, y in a) / n\n    pe = p1 * p2 + (1 - p1) * (1 - p2)\n    return {'factor': factor, 'n': n, 'owl_yes_rate': p1, 'qwen_yes_rate': p2, 'agreement': po,\n            'kappa': (po - pe) / (1 - pe) if pe < 1 else None,\n            'qwen_true_owl_false': sum(y and not x for x, y in a), 'owl_true_qwen_false': sum(x and not y for x, y in a)}\n\n\ndef model_diff(sd15, sdxl, prompts, factor):\n    keys = [k for k in sd15 if k in sdxl and factor in sd15[k] and sd15[k].get(factor) is not None and sdxl[k].get(factor) is not None]\n    x = [sd15[k][factor] for k in keys]\n    y = [sdxl[k][factor] for k in keys]\n    b_ = sum(a and not c for a, c in zip(x, y))\n    c_ = sum(c and not a for a, c in zip(x, y))\n    return {'factor': factor, 'paired_images': len(keys), 'sd15': acc(x), 'sdxl': acc(y), 'p': mcnemar(b_, c_)}\n\n\ndef run(sd15_root, sdxl_root):\n    res = {}\n    data = {'sd15': load(sd15_root), 'sdxl': load(sdxl_root)}\n    labs = {}\n    for m, (prompts, labels, pairs) in data.items():\n        for ev in ['owl', 'qwen']:\n            lab = variant(prompts, labels, ev)\n            labs[(m, ev)] = lab\n            res[(m, ev)] = {'attributes': attributes(lab), 'complexity': by_complexity(lab, prompts),\n                            'interference': interference(lab, prompts, pairs)}\n    res['agreement'] = {m: [agreement(labs[(m, 'owl')], labs[(m, 'qwen')], f) for f in ['N', 'P']] for m in data}\n    prompts = data['sd15'][0]\n    res['model_diff'] = {ev: [model_diff(labs[('sd15', ev)], labs[('sdxl', ev)], prompts, f) for f in ['N', 'P']] + [\n        dict(model_diff({k: {'S': v} for k, v in strict(labs[('sd15', ev)], prompts).items()},\n                        {k: {'S': v} for k, v in strict(labs[('sdxl', ev)], prompts).items()}, prompts, 'S'), factor='strict')]\n        for ev in ['owl', 'qwen']}\n    return res\n")
sys.path.insert(0, str(TOOLS))
import importlib, ablation
importlib.reload(ablation)

def get_root(name, path):
    path = Path(path)
    if all((path / f).exists() for f in ["labels.csv", "prompts.csv", "pairs.csv"]):
        print(name, "->", path)
        return path
    local = Path("/content") / name
    local.mkdir(exist_ok=True)
    print(f"{name}: {path} not found. Upload labels.csv, prompts.csv and pairs.csv of {name}.")
    for fname, data in files.upload().items():
        for target in ["labels", "prompts", "pairs"]:
            if fname.startswith(target):
                (local / f"{target}.csv").write_bytes(data)
    assert all((local / f).exists() for f in ["labels.csv", "prompts.csv", "pairs.csv"]), "Missing files"
    return local

SD15 = get_root("main_sd15", "/content/drive/MyDrive/attribute_interference/v1/main_sd15")
SDXL = get_root("main_sdxl", "/content/drive/MyDrive/sdxl_v1/main_sdxl 2")
for name, root in [("SD 1.5", SD15), ("SDXL", SDXL)]:
    lab = pd.read_csv(root / "labels.csv")
    print(f"{name}: {len(lab)} images | qwen ok {(lab.qwen_status == 'ok').sum()} | owl ok {(lab.owl_status == 'ok').sum()}")

Mounted at /content/drive
main_sd15 -> /content/drive/MyDrive/attribute_interference/v1/main_sd15
main_sdxl -> /content/drive/MyDrive/sdxl_v1/main_sdxl 2
SD 1.5: 2301 images | qwen ok 2301 | owl ok 1727
SDXL: 2304 images | qwen ok 2304 | owl ok 1728


## 2. Compute both evaluator variants

In [ ]:
res = ablation.run(SD15, SDXL)
MODELS = {"sd15": "SD 1.5", "sdxl": "SDXL"}
EVALS = {"owl": "OWLv2", "qwen": "Qwen"}
pct = lambda v: None if v is None or pd.isna(v) else round(100 * v, 1)
print("Done")

Done


## 3. Count and position accuracy

In [ ]:
rows = []
for m in MODELS:
    for f in ["N", "P"]:
        row = {"model": MODELS[m], "factor": f}
        for ev in EVALS:
            a = next(r for r in res[(m, ev)]["attributes"] if r["factor"] == f)
            row[f"{EVALS[ev]} acc %"] = pct(a["accuracy"])
            row[f"{EVALS[ev]} singleton %"] = pct(a["singleton_accuracy"])
        rows.append(row)
acc_table = pd.DataFrame(rows)
display(acc_table)

,model,factor,OWLv2 acc %,OWLv2 singleton %,Qwen acc %,Qwen singleton %
0,SD 1.5,N,22.2,33.8,30.8,46.2
1,SD 1.5,P,19.7,28.7,3.6,10.0
2,SDXL,N,19.0,36.2,33.8,48.8
3,SDXL,P,22.2,48.8,4.3,10.0


## 4. Agreement between OWLv2 and Qwen labels
Images where the object is present and both labels are known.

In [ ]:
agree = pd.DataFrame([{"model": MODELS[m], **r} for m in MODELS for r in res["agreement"][m]])
for c in ["owl_yes_rate", "qwen_yes_rate", "agreement"]:
    agree[c] = agree[c].map(pct)
agree["kappa"] = agree["kappa"].round(3)
display(agree)

,model,factor,n,owl_yes_rate,qwen_yes_rate,agreement,kappa,qwen_true_owl_false,owl_true_qwen_false
0,SD 1.5,N,1057,24.2,33.5,75.6,0.412,178,80
1,SD 1.5,P,1014,22.4,4.0,79.9,0.183,9,195
2,SDXL,N,1137,19.3,34.2,75.9,0.402,222,52
3,SDXL,P,1117,22.9,4.5,78.9,0.166,15,221


## 5. SDXL vs SD 1.5 under each evaluator
Paired over identical prompt-seed jobs; exact McNemar p.

In [ ]:
rows = []
for ev in EVALS:
    for r in res["model_diff"][ev]:
        rows.append({"evaluator": EVALS[ev], "metric": r["factor"], "images": r["paired_images"],
                     "SD 1.5 %": pct(r["sd15"]), "SDXL %": pct(r["sdxl"]),
                     "diff pp": pct(r["sdxl"] - r["sd15"]), "McNemar p": round(r["p"], 4)})
diff_table = pd.DataFrame(rows)
display(diff_table)

,evaluator,metric,images,SD 1.5 %,SDXL %,diff pp,McNemar p
0,OWLv2,N,1151,22.2,19.0,-3.2,0.0443
1,OWLv2,P,1152,19.7,22.2,2.5,0.1145
2,OWLv2,strict,2301,18.4,27.9,9.5,0.0000
3,Qwen,N,1151,30.8,33.8,3.0,0.1108
4,Qwen,P,1152,3.6,4.3,0.8,0.3800
5,Qwen,strict,2301,16.1,24.5,8.4,0.0000


## 6. Strict accuracy by number of active factors

In [ ]:
rows = []
for i, c in enumerate(res[("sd15", "owl")]["complexity"]):
    row = {"complexity": c["complexity"]}
    for m in MODELS:
        for ev in EVALS:
            row[f"{MODELS[m]} {EVALS[ev]} %"] = pct(res[(m, ev)]["complexity"][i]["strict_accuracy"])
    rows.append(row)
complexity_table = pd.DataFrame(rows)
display(complexity_table)

,complexity,SD 1.5 OWLv2 %,SD 1.5 Qwen %,SDXL OWLv2 %,SDXL Qwen %
0,0,100.0,100.0,100.0,100.0
1,1,46.4,45.1,66.7,61.2
2,2,18.1,14.1,30.7,27.2
3,3,4.8,2.0,10.8,6.8
4,4,0.3,0.3,2.1,0.6
5,5,0.0,0.0,0.0,0.0
6,all,18.4,16.1,28.0,24.5


## 7. Interference involving count and position
Δ(A, B): change in accuracy of A after adding B. Only rows where the checked factor is N or P differ between evaluators.

In [ ]:
rows = []
for m in MODELS:
    owl_rows = {(r["checked"], r["added"]): r for r in res[(m, "owl")]["interference"]}
    qwen_rows = {(r["checked"], r["added"]): r for r in res[(m, "qwen")]["interference"]}
    for key in owl_rows:
        if key[0] not in {"N", "P"}:
            continue
        o, q = owl_rows[key], qwen_rows[key]
        fmt = lambda r: None if r["delta"] is None else f"{r['delta']:+.2f} [{r['ci_low']:+.2f}, {r['ci_high']:+.2f}]"
        rows.append({"model": MODELS[m], "checked": key[0], "added": key[1],
                     "OWLv2 delta [95% CI]": fmt(o), "OWLv2 Holm p": round(o["p_holm"], 3),
                     "Qwen delta [95% CI]": fmt(q), "Qwen Holm p": round(q["p_holm"], 3),
                     "pairs OWLv2 / Qwen": f"{o['pairs']} / {q['pairs']}"})
inter_table = pd.DataFrame(rows)
display(inter_table)

,model,checked,added,OWLv2 delta [95% CI],OWLv2 Holm p,Qwen delta [95% CI],Qwen Holm p,pairs OWLv2 / Qwen
0,SD 1.5,N,C,"-0.05 [-0.11, +0.02]",1.000,"-0.04 [-0.12, +0.03]",1.000,498 / 498
1,SD 1.5,N,P,"-0.00 [-0.04, +0.04]",1.000,"-0.06 [-0.10, -0.02]",0.408,502 / 502
2,SD 1.5,N,M,"+0.00 [-0.05, +0.06]",1.000,"+0.01 [-0.06, +0.07]",1.000,438 / 438
3,SD 1.5,N,O2,"-0.04 [-0.10, +0.02]",1.000,"-0.04 [-0.11, +0.03]",1.000,486 / 486
4,SD 1.5,P,C,"+0.03 [-0.03, +0.08]",1.000,"-0.01 [-0.04, +0.02]",1.000,465 / 465
5,SD 1.5,P,N,"-0.31 [-0.38, -0.24]",0.000,"-0.05 [-0.09, -0.01]",0.002,467 / 467
6,SD 1.5,P,M,"-0.00 [-0.05, +0.05]",1.000,"-0.02 [-0.05, +0.00]",0.699,410 / 410
7,SD 1.5,P,O2,"+0.05 [+0.01, +0.09]",0.410,"-0.03 [-0.05, -0.01]",0.428,449 / 449
8,SDXL,N,C,"-0.06 [-0.10, -0.03]",0.020,"-0.05 [-0.11, +0.01]",0.739,562 / 562
9,SDXL,N,P,"-0.04 [-0.07, -0.01]",0.389,"-0.03 [-0.08, +0.01]",1.000,564 / 564


## 8. Do the conclusions change?

In [ ]:
checks = []
for ev in EVALS:
    d = {r["factor"]: r for r in res["model_diff"][ev]}
    for f in ["N", "P"]:
        checks.append({"conclusion": f"SDXL does not improve {f} over SD 1.5 (p >= 0.05)", "evaluator": EVALS[ev],
                       "holds": d[f]["p"] >= 0.05, "detail": f"{pct(d[f]['sd15'])}% -> {pct(d[f]['sdxl'])}%, p = {d[f]['p']:.3g}"})
    checks.append({"conclusion": "SDXL strict accuracy higher than SD 1.5 (p < 0.05)", "evaluator": EVALS[ev],
                   "holds": d["strict"]["p"] < 0.05 and d["strict"]["sdxl"] > d["strict"]["sd15"],
                   "detail": f"{pct(d['strict']['sd15'])}% -> {pct(d['strict']['sdxl'])}%, p = {d['strict']['p']:.3g}"})
    for f in ["N", "P"]:
        s = next(r for r in res[("sdxl", ev)]["attributes"] if r["factor"] == f)["singleton_accuracy"]
        checks.append({"conclusion": f"SDXL {f} singleton accuracy below the 60% criterion", "evaluator": EVALS[ev],
                       "holds": s is not None and s < 0.6, "detail": f"{pct(s)}%"})
    for m in MODELS:
        inter = [r for r in res[(m, ev)]["interference"] if r["delta"] is not None]
        ranked = sorted(inter, key=lambda r: r["delta"])
        pn = next(r for r in inter if r["checked"] == "P" and r["added"] == "N")
        rank = ranked.index(pn) + 1
        sig = sum(r["p_holm"] < 0.05 for r in inter)
        checks.append({"conclusion": f"{MODELS[m]}: adding N is the strongest drop for P (rank 1 of 20)", "evaluator": EVALS[ev],
                       "holds": rank == 1, "detail": f"delta {pn['delta']:+.2f}, rank {rank}, Holm p = {pn['p_holm']:.3g}; {sig} significant pairs"})
check_table = pd.DataFrame(checks)
display(check_table)

,conclusion,evaluator,holds,detail
0,SDXL does not improve N over SD 1.5 (p >= 0.05),OWLv2,False,"22.2% -> 19.0%, p = 0.0443"
1,SDXL does not improve P over SD 1.5 (p >= 0.05),OWLv2,True,"19.7% -> 22.2%, p = 0.115"
2,SDXL strict accuracy higher than SD 1.5 (p < 0...,OWLv2,True,"18.4% -> 27.9%, p = 4.4e-24"
3,SDXL N singleton accuracy below the 60% criterion,OWLv2,True,36.2%
4,SDXL P singleton accuracy below the 60% criterion,OWLv2,True,48.8%
5,SD 1.5: adding N is the strongest drop for P (...,OWLv2,True,"delta -0.31, rank 1, Holm p = 9.3e-30; 6 signi..."
6,SDXL: adding N is the strongest drop for P (ra...,OWLv2,True,"delta -0.38, rank 1, Holm p = 1.59e-48; 8 sign..."
7,SDXL does not improve N over SD 1.5 (p >= 0.05),Qwen,True,"30.8% -> 33.8%, p = 0.111"
8,SDXL does not improve P over SD 1.5 (p >= 0.05),Qwen,True,"3.6% -> 4.3%, p = 0.38"
9,SDXL strict accuracy higher than SD 1.5 (p < 0...,Qwen,True,"16.1% -> 24.5%, p = 1.43e-23"


## 9. Save

In [ ]:
OUT = Path("/content/drive/MyDrive/sdxl_v1/evaluator_ablation_count_position")
OUT.mkdir(exist_ok=True)
for name, df in [("accuracy", acc_table), ("owl_qwen_agreement", agree), ("sdxl_vs_sd15", diff_table),
                 ("strict_by_complexity", complexity_table), ("interference_n_p", inter_table), ("conclusion_checks", check_table)]:
    df.to_csv(OUT / f"{name}.csv", index=False)
full = []
for (m, ev), v in ((k, v) for k, v in res.items() if isinstance(k, tuple)):
    for r in v["interference"]:
        full.append({"model": MODELS[m], "evaluator": EVALS[ev], **r})
pd.DataFrame(full).to_csv(OUT / "interference_all.csv", index=False)
print("Saved to", OUT, sorted(p.name for p in OUT.iterdir()))

Saved to /content/drive/MyDrive/sdxl_v1/evaluator_ablation_count_position ['accuracy.csv', 'conclusion_checks.csv', 'interference_all.csv', 'interference_n_p.csv', 'owl_qwen_agreement.csv', 'sdxl_vs_sd15.csv', 'strict_by_complexity.csv']
